In [1]:
# ============================================================
# HEALTHCARE DATA: NLP + K-MEANS CLUSTERING
# ============================================================

# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy.sparse import hstack, csr_matrix

# ============================================================
# 2. LOAD DATA
# ============================================================

df = pd.read_csv("cleaned_healthcare_messy_data.csv")

print("Shape:", df.shape)
display(df.head())

# ============================================================
# 3. BASIC CLEANING
# ============================================================

df.columns = (
    df.columns.str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

df = df.drop_duplicates()

# Fill missing values
for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna("unknown")

print("Missing values:", df.isnull().sum().sum())

# ============================================================
# 4. FIND TEXT AND NUMERICAL COLUMNS
# ============================================================

numeric_cols = df.select_dtypes(
    include=np.number
).columns.tolist()

text_cols = df.select_dtypes(
    include="object"
).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Text columns:", text_cols)

# ============================================================
# 5. NLP - COMBINE TEXT COLUMNS
# ============================================================

if text_cols:

    df["text"] = df[text_cols].astype(str).agg(
        " ".join, axis=1
    )

else:
    df["text"] = ""

# Clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_text"] = df["text"].apply(clean_text)

# ============================================================
# 6. TF-IDF
# ============================================================

tfidf = TfidfVectorizer(
    max_features=1000,
    stop_words="english"
)

X_text = tfidf.fit_transform(
    df["clean_text"]
)

print("TF-IDF shape:", X_text.shape)

# ============================================================
# 7. NUMERICAL FEATURES
# ============================================================

if numeric_cols:

    scaler = StandardScaler()

    X_num = scaler.fit_transform(
        df[numeric_cols]
    )

    X_num = csr_matrix(X_num)

    # Combine NLP + numerical data
    X = hstack([
        X_num,
        X_text
    ])

else:

    X = X_text

print("Final feature shape:", X.shape)

# ============================================================
# 8. FIND BEST K
# ============================================================

scores = {}

for k in range(2, 7):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X)

    scores[k] = silhouette_score(
        X,
        labels
    )

print("\nSilhouette Scores:")
for k, score in scores.items():
    print(f"K={k}: {score:.3f}")

best_k = max(
    scores,
    key=scores.get
)

print("\nBest K:", best_k)

# ============================================================
# 9. FINAL K-MEANS
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

# ============================================================
# 10. CLUSTER RESULTS
# ============================================================

print("\nCluster Distribution:")
print(df["cluster"].value_counts().sort_index())

display(
    df.groupby("cluster").size()
    .reset_index(name="records")
)

# ============================================================
# 11. TOP NLP WORDS IN EACH CLUSTER
# ============================================================

words = tfidf.get_feature_names_out()

for cluster in range(best_k):

    rows = df["cluster"] == cluster

    cluster_text = X_text[rows]

    mean_scores = np.asarray(
        cluster_text.mean(axis=0)
    ).flatten()

    top_words = mean_scores.argsort()[::-1][:10]

    print(
        f"\nCluster {cluster} top words:"
    )

    print(
        ", ".join(words[top_words])
    )

# ============================================================
# 12. VISUALIZE CLUSTERS
# ============================================================

plt.figure(figsize=(8, 5))

plt.bar(
    df["cluster"].value_counts().sort_index().index,
    df["cluster"].value_counts().sort_index().values
)

plt.xlabel("Cluster")
plt.ylabel("Number of Records")
plt.title("Healthcare K-Means Clusters")
plt.show()

# ============================================================
# 13. SAVE RESULT
# ============================================================

df.to_csv(
    "healthcare_kmeans_nlp_result.csv",
    index=False
)

print(
    "\nSaved as: healthcare_kmeans_nlp_result.csv"
)

Shape: (1000, 10)


,Patient Name,Age,Gender,Condition,Medication,Visit Date,Blood Pressure,Cholesterol,Email,Phone Number
0,david lee,25,Other,Heart Disease,METFORMIN,2020-01-15,140/90,200,name@hospital.org,555-555-5555
1,emily davis,0,Male,Diabetes,NONE,NaN,120/80,200,no_email@example.com,000-000-0000
2,laura martinez,35,Other,Asthma,METFORMIN,NaN,110/70,160,contact@domain.com,000-000-0000
3,michael wilson,0,Male,Diabetes,ALBUTEROL,2020-01-15,110/70,0,name@hospital.org,555-555-5555
4,david lee,0,Female,Asthma,NONE,NaN,110/70,180,no_email@example.com,NaN


Missing values: 0
Numeric columns: ['age', 'cholesterol']
Text columns: ['patient_name', 'gender', 'condition', 'medication', 'visit_date', 'blood_pressure', 'email', 'phone_number']
TF-IDF shape: (1000, 40)
Final feature shape: (1000, 42)


/var/folders/88/w4w1n8l12kd_z42_6mrnndmw0000gn/T/ipykernel_4662/4043908857.py:42: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
/var/folders/88/w4w1n8l12kd_z42_6mrnndmw0000gn/T/ipykernel_4662/4043908857.py:55: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pand


Silhouette Scores:
K=2: 0.383
K=3: 0.311
K=4: 0.283
K=5: 0.281
K=6: 0.276

Best K: 2

Cluster Distribution:
cluster
0    769
1    231
Name: count, dtype: int64


,cluster,records
0,0,769
1,1,231


AttributeError: 'Series' object has no attribute 'nonzero'